# 한국 배당성장주 최종 통합 백테스트 · 10년 PIT & 15년 장기 추정 (Google Colab)

**목적:** 1,252개사 풀 유니버스 PIT 검증과 버그 감사를 통해 최종 선정된 **바구니 A 개선 규칙**과 **A+B+C 3바구니 통합 포트폴리오(`50/20/30` vs `40/15/45`)**를 **거치식**과 **실전 월별 적립식(매도규칙 준수 장기보유)**으로 한 번에 검증합니다.

### 기존 `colab_alpha_lab` 대비 반영된 핵심 개선사항
1. **엔진 버그 4종 완전 보정:**
   - ① `v6`(자사주·희석) 액면분할 보정(`split_factor_to_latest` 반영)
   - ② `FY2025`(`2026-04`) 리밸런싱 및 `FY2025` 배당금 정상 수취 반영
   - ③ 현행 전략 A에도 비교 전략과 동일한 유동성 필터(60일 거래대금 ≥ 10억 원) 적용
   - ④ 순이익비 총환원율 대신 **시가총액 대비 진성 주주환원수익률 `(배당+소각)/시총`** 반영
2. **최종 선정 바구니 A 규칙 (생존 DNA 고도화):**
   - `1~8번` 품질 필터 통과 풀 내부에서 **저변동성(30%) + 절대밸류 PER·PBR(30%) + 배당수익률(20%) + 진성 주주환원수익률(20%)** 순위 적용
   - **고정 `35%` 변동성 게이트의 급락장 매수 마비 해소:** 상위 30% 고변동 테마주만 거르는 **상대 70% 변동성 게이트** 채택
   - **금융 업종 상한 반영:** 바구니 A 내 금융 비중 ≤ 50%(전체 포트폴리오 금융 상한 35% 안전 준수)
3. **3바구니 통합 포트폴리오 (`docs/운용규칙.md` 2·3·5절 재현):**
   - **A:** 국내 배당성장 (1,252 클린 PIT)
   - **B:** 반도체·AI (`005935` 삼성전자우 TR 50% + `SOXX` 원화 환노출 50%)
   - **C:** 미국 지수 2:1 (`SPY` 2/3 + `SCHD` 1/3 원화 환노출, 보수·배당세 보수적 차감)
   - **적립식(매도 없음):** 매월 200만 원 적립, 운용규칙 5절대로 허용폭·목표 미달 바구니에만 신규 자금 투입
4. **검증 기간 3단계 확장 (DART API 키 불필요 — 기존 Drive 1,252 PIT DB 자동 활용):**
   - **공식 7.7년 PIT (`2019-04 ~ 최신`, `FY2018~FY2025`):** 기존 연구와 동일 기준선 대조
   - **확장 9.7년 PIT (`2017-04 ~ 최신`, `FY2016~FY2025`):** 추가 DART 호출 없이 기존 PIT DB의 `2014~2017` 재무·배당 + 네이버 `fchart(3000봉, 2014-09~)` 활용 *(2018 미중 무역분쟁 하락장 포함)*
   - **초장기 14.7년 추정 (`2012-04 ~ 최신`, `SCHD` 상장 이후 전 구간):** `2012~2017` 구간은 B·C 100% 실측 + A는 ① **보수적 KOSPI TR 연결(생존편향 추가 0%)** ② **Yahoo 20년 배당·저변동 프록시 연결(참고 추정)** 2종으로 분리 검증

> **주의 (과거 확장 구간의 정직한 한계):** OpenDART XBRL 구조화 재무는 `FY2015`(비교연도 `2013~2014`)부터 제공되므로 `2012~2016` 추정 구간에는 재무 3~8번(FCF·ROIC·소각)이 반영되지 않으며, 과거로 갈수록 상장폐지 생존편향이 커집니다. 따라서 리포트에서 **정밀 PIT 구간(`2017~2026`, `2019~2026`)**과 **추정 참고 구간(`2012~2017`)**을 반드시 분리해 해석하세요.


In [ ]:
# ==============================================================================
# 1. 환경 설정 · 한글 폰트(fonts-nanum) 및 PDF 도구 자동 설치 (API 키 불필요)
# ==============================================================================
import os, sys, re, io, gzip, json, math, time, base64, warnings, statistics
from pathlib import Path
from datetime import datetime, timezone, timedelta
from html import escape
from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[환경] Google Colab 감지 -> 한글 폰트(fonts-nanum) 및 PDF 엔진 설치 중...")
    subprocess.run(["apt-get", "-qq", "update"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-nanum", "libpango-1.0-0", "libpangoft2-1.0-0"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "weasyprint"], check=False)

import numpy as np
import pandas as pd
import requests
import matplotlib
if not IN_COLAB:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as mtick

try:
    from IPython.display import display
except ImportError:
    display = print

def setup_korean_font():
    chosen = None
    for p in [
        "/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
        "/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf",
    ]:
        if os.path.exists(p):
            fm.fontManager.addfont(p)
            if chosen is None:
                chosen = fm.FontProperties(fname=p).get_name()
    if chosen is None:
        avail = {f.name for f in fm.fontManager.ttflist}
        for cand in ["NanumGothic", "AppleGothic", "Malgun Gothic", "DejaVu Sans"]:
            if cand in avail:
                chosen = cand
                break
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except Exception:
        pass
    if chosen:
        plt.rcParams["font.family"] = chosen
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams["figure.dpi"] = 120
    return chosen

FONT_NAME = setup_korean_font()
print(f"[완료] 한글 폰트: {FONT_NAME} | Python {sys.version.split()[0]} | Colab={IN_COLAB}")

In [ ]:
# ==============================================================================
# 2. 최종 백테스트 설정 · Google Drive 연결 · 체크포인트 이어받기
# ==============================================================================
FINAL_CONFIG = {
    # 검증 기간 설정
    "EXT_START_FY": 2016,          # 확장 9.7년 PIT 시작 (FY2016 -> 2017-04 출발, 추가 DART 호출 0!)
    "OFFICIAL_START_FY": 2018,     # 공식 7.7년 PIT 시작 (FY2018 -> 2019-04 출발, 기존 기준선 대조용)
    "END_FY": 2025,                # 최신 확정 사업연도 (FY2025 -> 2026-04 리밸런싱 및 배당 포함!)
    "ENABLE_15Y_ESTIMATE": True,   # 2012-04~최신(14.7년, SCHD 상장 이후) 장기 추정 백테스트 실행 여부

    # 바구니 A 최종 선정 규칙 파라미터
    "TOP_N": 10,
    "PIT_MIN_MCAP_EOK": 3000.0,    # 매수 시점 PIT 시총 하한 (억 원)
    "MIN_LIQ_EOK": 10.0,           # 60일 일평균 거래대금 하한 (억 원, 전 전략 동일 적용)
    "VAL_CAP": 0.50,               # 상대 밸류 괴리율 ±50% 캡
    "MAX_ABS_PER": 15.0,           # 절대밸류 상한 캡 (PER)
    "MAX_ABS_PBR": 1.8,            # 절대밸류 상한 캡 (비금융 PBR)
    "REL_VOL_QUANTILE": 0.70,      # 최종 선정: 상대 70% 변동성 게이트 (상위 30% 고변동 테마주만 제외)
    "FIXED_VOL_CAP": 0.35,         # 비교용: 기존 고정 35% 변동성 게이트
    "MAX_FIN_RATIO_IN_A": 0.50,    # 바구니 A 내 금융 업종 비중 상한 (50% -> 전체 포트폴리오 20~25%, 35% 상한 안전 준수)

    # 비용 · 세금 · 적립식 설정
    "INITIAL_CAPITAL": 100_000_000.0,
    "MONTHLY_DCA_A_KRW": 1_000_000.0,      # 바구니 A 단독 적립식 월 투입금 (100만 원)
    "MONTHLY_DCA_PORT_KRW": 2_000_000.0,   # 통합 3바구니 적립식 월 투입금 (200만 원)
    "FEE_RATE": 0.0025,                    # 개별주 편도 거래비용 0.25%
    "PORT_FEE_RATE": 0.0020,               # 통합 바구니 편도 비용 0.20%
    "DIV_TAX_RATE": 0.154,                 # 국내 배당소득세 15.4%
    "KOSPI_EST_DIV_YIELD": 0.018,          # KOSPI TR 추정 세후 배당 가산 (+1.8%/년)

    # Google Drive 경로 (기존 dividend_alpha_lab 풀 DB 및 캐시 자동 공유!)
    "USE_GOOGLE_DRIVE": True,
    "DRIVE_DIR": "/content/drive/MyDrive/Colab Notebooks/dividend_alpha_lab",
    "FULL_PIT_FILENAME": "full_kr_fundamentals_pit.jsonl.gz",
    "FORCE_REFRESH_PRICES": False,

    # 공개 데이터 소스 (Common_Everything Public)
    "GITHUB_REPO": "khs0221/Common_Everything",
    "GITHUB_BRANCHES": ["feat/dividend-backtest", "main", "copilot/ai-coding-curriculum"],
    "DATA_SUBPATH": "dividend-backtest/data",
    "CACHE_DIR": "./_cache_final_bt",
    "WORKERS": 10,
}

CACHE_DIR = Path(FINAL_CONFIG["CACHE_DIR"])
CACHE_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_ROOT = None
DRIVE_REPORT_DIR = None
if IN_COLAB and FINAL_CONFIG["USE_GOOGLE_DRIVE"]:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = Path(FINAL_CONFIG["DRIVE_DIR"])
        DRIVE_REPORT_DIR = DRIVE_ROOT / "reports"
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
        DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"[Drive 연결 완료] DB·리포트 경로: {DRIVE_ROOT}")
    except Exception as e:
        print(f"[Drive 미연결] 로컬 저장소 사용 ({type(e).__name__}: {e})")

def save_ckpt_json(filename: str, obj: dict) -> None:
    raw = gzip.compress(json.dumps(obj, ensure_ascii=False, separators=(",", ":")).encode("utf-8"), compresslevel=4)
    (CACHE_DIR / filename).write_bytes(raw)
    if DRIVE_ROOT is not None:
        try:
            tmp = DRIVE_ROOT / (filename + ".tmp")
            dst = DRIVE_ROOT / filename
            tmp.write_bytes(raw)
            tmp.replace(dst)
        except Exception as e:
            print(f"  [주의] Drive 체크포인트 저장 지연({type(e).__name__})")

def load_ckpt_json(filename: str, force: bool = False) -> dict:
    if force:
        return {}
    search_dirs = [DRIVE_ROOT, CACHE_DIR] if DRIVE_ROOT is not None else [CACHE_DIR]
    for base in search_dirs:
        p = base / filename
        if p.exists() and p.stat().st_size > 0:
            try:
                return json.loads(gzip.decompress(p.read_bytes()).decode("utf-8"))
            except Exception:
                continue
    return {}

print(f"[설정 완료] 확장 PIT FY{FINAL_CONFIG['EXT_START_FY']}~FY{FINAL_CONFIG['END_FY']} (2017-04~) | "
      f"공식 PIT FY{FINAL_CONFIG['OFFICIAL_START_FY']}~FY{FINAL_CONFIG['END_FY']} (2019-04~) | "
      f"15년 추정={FINAL_CONFIG['ENABLE_15Y_ESTIMATE']}")

In [ ]:
# ==============================================================================
# 3. 1,252개사 PIT DB 자동 로드 + 3,000일(2014-09~) 일별 수정주가 이어받기 수집
# ==============================================================================
def load_public_data_bytes(filename: str) -> bytes:
    for p in [
        Path("../data") / filename,
        Path("dividend-backtest/data") / filename,
        Path("/tmp/review") / filename,
        CACHE_DIR / filename,
    ]:
        if p.exists():
            return p.read_bytes()
    for br in FINAL_CONFIG["GITHUB_BRANCHES"]:
        url = f"https://raw.githubusercontent.com/{FINAL_CONFIG['GITHUB_REPO']}/{br}/{FINAL_CONFIG['DATA_SUBPATH']}/{filename}"
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 200 and len(r.content) > 100:
                (CACHE_DIR / filename).write_bytes(r.content)
                return r.content
        except Exception:
            continue
    raise RuntimeError(f"공개 데이터({filename})를 찾을 수 없습니다.")

def _convert_374_to_pit(rec: dict) -> dict:
    if "pit" in rec:
        return rec
    pit_out = {}
    for fy_k, cols in (rec.get("snapshots") or {}).items():
        if not isinstance(cols, dict):
            continue
        by_y = {}
        for col_name, y_map in cols.items():
            if not isinstance(y_map, dict):
                continue
            for yk, val in y_map.items():
                if val is not None:
                    by_y.setdefault(str(yk), {})[col_name] = val
        pit_out[str(fy_k)] = by_y
    return {
        "ticker": rec["ticker"], "name": rec["name"], "market": rec.get("market", "KOSPI"),
        "path": rec.get("path", "general"), "act_dps": rec.get("act_dps") or {}, "pit": pit_out,
    }

base374_raw = gzip.decompress(load_public_data_bytes("kr_fundamentals_pit.jsonl.gz")).decode("utf-8")
BASE374_MAP = {r["ticker"]: _convert_374_to_pit(r) for r in (json.loads(l) for l in base374_raw.splitlines() if l.strip())}

full_pit_candidates = [
    (DRIVE_ROOT / FINAL_CONFIG["FULL_PIT_FILENAME"]) if DRIVE_ROOT is not None else None,
    Path("/tmp/review") / FINAL_CONFIG["FULL_PIT_FILENAME"],
    CACHE_DIR / FINAL_CONFIG["FULL_PIT_FILENAME"],
]
full_pit_path = next((p for p in full_pit_candidates if p is not None and p.exists() and p.stat().st_size > 1000), None)

if full_pit_path is not None:
    pit_raw = gzip.decompress(full_pit_path.read_bytes()).decode("utf-8")
    full_list = [json.loads(line) for line in pit_raw.splitlines() if line.strip()]
    enriched_cnt = 0
    for rec in full_list:
        b = BASE374_MAP.get(rec["ticker"])
        if not b:
            continue
        enriched_cnt += 1
        for yk, dv in (b.get("act_dps") or {}).items():
            if dv is not None:
                rec.setdefault("act_dps", {})[str(yk)] = dv
        for fy_k, b_frame in (b.get("pit") or {}).items():
            t_frame = rec.setdefault("pit", {}).setdefault(str(fy_k), {})
            for yk, b_cols in b_frame.items():
                t_cols = t_frame.setdefault(str(yk), {})
                for col_k, col_v in b_cols.items():
                    if t_cols.get(col_k) is None and col_v is not None:
                        t_cols[col_k] = col_v
    seen_tks = {r["ticker"] for r in full_list}
    for tk, b in BASE374_MAP.items():
        if tk not in seen_tks:
            full_list.append(b)
    COMPANIES = full_list
    DATASET_LABEL = f"풀 유니버스 PIT DB ({len(COMPANIES):,}개사)"
    print(f"[풀 유니버스 자동 로드] {full_pit_path.name} ({len(COMPANIES):,}개사, 374 금융/XBRL {enriched_cnt}개사 보강)")
else:
    COMPANIES = list(BASE374_MAP.values())
    DATASET_LABEL = f"기본 PIT 표본 ({len(COMPANIES):,}개사)"
    print(f"[안내] Drive에 풀 DB가 없어 기본 {len(COMPANIES)}개사 표본으로 실행합니다.")

rf_df = pd.read_csv(io.StringIO(load_public_data_bytes("kr_treasury_10y_annual.csv").decode("utf-8")))
RF_MAP = {int(r["연도"]): float(r["국고채10년_연평균_퍼센트"]) / 100.0 for _, r in rf_df.iterrows()}

# 네이버 fchart 3,000일 수집 (2014-09-04 ~ 최신, 확장 9.7년 PIT 백테스트 완벽 지원!)
_FCHART_URL = "https://fchart.stock.naver.com/sise.nhn"
_ITEM_RE = re.compile(r'<item data="([^"]+)"')
_HEADERS = {"User-Agent": "Mozilla/5.0 (dividend-final-bt)"}

def fetch_naver_ohlcv_3000(symbol: str, count: int = 3000, min_ymd: str = "20140901") -> list[list]:
    for attempt in range(3):
        try:
            r = requests.get(
                _FCHART_URL,
                params={"symbol": symbol, "timeframe": "day", "count": count, "requestType": 0},
                headers=_HEADERS,
                timeout=20,
            )
            r.raise_for_status()
            out = []
            for m in _ITEM_RE.finditer(r.content.decode("euc-kr", errors="ignore")):
                parts = m.group(1).split("|")
                c = float(parts[4])
                v = float(parts[5])
                if c > 0 and parts[0] >= min_ymd:
                    out.append([parts[0], c, v])
            if out:
                return out
        except Exception:
            time.sleep(0.35 * (attempt + 1))
    return []

KST = timezone(timedelta(hours=9))
today_tag = datetime.now(KST).strftime("%Y-%m-%d")
CKPT_PX_NAME = f"ckpt_prices_3000d_{today_tag[:7]}_{len(COMPANIES)}.json.gz"
px_ckpt = load_ckpt_json(CKPT_PX_NAME, force=FINAL_CONFIG["FORCE_REFRESH_PRICES"])
px_store = px_ckpt.get("data", {})

# 로컬 테스트용 사전 캐시 폴백 (날짜 범위가 2014~를 덮지 않으면 3000봉 수집으로 보완)
all_symbols = ["KOSPI", "005935"] + [c["ticker"] for c in COMPANIES]
all_symbols = list(dict.fromkeys(all_symbols))
todo_syms = [s for s in all_symbols if s not in px_store]

if px_store and not todo_syms:
    print(f"[시세 캐시 복원 완료] {len(px_store):,}개 종목 3,000일 일봉 즉시 로드!")
else:
    print(f"[시세 수집] 보유 {len(px_store):,}개 -> 남은 {len(todo_syms):,}개 수집 중 (40종목마다 Drive 저장)...")
    t0 = time.time()
    try:
        with ThreadPoolExecutor(max_workers=FINAL_CONFIG["WORKERS"]) as ex:
            futs = {ex.submit(fetch_naver_ohlcv_3000, s): s for s in todo_syms}
            for idx, fut in enumerate(as_completed(futs), 1):
                sym = futs[fut]
                rows = fut.result()
                if rows:
                    px_store[sym] = rows
                if idx % 40 == 0 or idx == len(futs):
                    save_ckpt_json(CKPT_PX_NAME, {"updated": today_tag, "data": px_store})
                    if idx % 200 == 0 or idx == len(futs):
                        print(f"  시세 진행 {len(px_store):,}/{len(all_symbols):,} ({time.time()-t0:.0f}초)")
    finally:
        if todo_syms:
            save_ckpt_json(CKPT_PX_NAME, {"updated": today_tag, "data": px_store})

def _to_df(rows: list[list]) -> pd.DataFrame:
    df = pd.DataFrame(rows, columns=["date", "close", "volume"])
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
    return df.set_index("date").sort_index()

KOSPI_DF = _to_df(px_store["KOSPI"])
PRICE_MAP = {t: _to_df(rows) for t, rows in px_store.items() if t != "KOSPI" and rows}
CALENDAR = KOSPI_DF.index
print(f"[완료] 일별 수정주가 {len(PRICE_MAP):,}개사 + KOSPI ({CALENDAR[0].date()} ~ {CALENDAR[-1].date()}, {len(CALENDAR):,} 거래일)")

In [ ]:
# ==============================================================================
# 4. 클린 PIT 품질 판정(버그 4종 보정) + 확장 9.7년(FY2016~FY2025) 팩터 패널 구축
# ==============================================================================
def get_val(frame: dict, col: str, y: int):
    v = (frame.get(str(y)) or {}).get(col)
    return None if v is None else float(v)

def cagr_avg(vals_early, vals_late, span: float, min_len: int = 2):
    if len(vals_early) < min_len or len(vals_late) < min_len:
        return None
    a, b = sum(vals_early) / len(vals_early), sum(vals_late) / len(vals_late)
    return (b / a) ** (1.0 / span) - 1.0 if (a > 0 and b > 0) else None

def get_pit_frame(comp: dict, fy: int) -> dict | None:
    """FY2018~2025는 해당 연도 PIT 스냅샷 그대로, FY2016~2017은 pit['2018']에서 <= fy만 엄격 슬라이싱."""
    pit_all = comp.get("pit") or {}
    if str(fy) in pit_all:
        return pit_all[str(fy)]
    if fy < 2018 and "2018" in pit_all:
        sliced = {yk: yv for yk, yv in pit_all["2018"].items() if int(yk) <= fy}
        return sliced if sliced else None
    return None

def extract_company_year_clean(comp: dict, fy: int, rebal_dt: pd.Timestamp, include_non_pass: bool = False) -> dict | None:
    t = comp["ticker"]
    px_df = PRICE_MAP.get(t)
    if px_df is None or px_df.empty:
        return None
    hist = px_df.loc[:rebal_dt]
    if len(hist) < 120:
        return None
    px_now = float(hist["close"].iloc[-1])
    if px_now <= 0:
        return None

    frame = get_pit_frame(comp, fy)
    if not frame:
        return None
    path = comp.get("path") or "general"
    act_dps = {int(k): float(v) for k, v in (comp.get("act_dps") or {}).items() if v is not None}

    # --- 1번 배당 이력 (FY2016~2017 조기 확장 구간은 3~4년 창 보정, FY2018+ 5년 창) ---
    win_len = 3 if fy == 2016 else (4 if fy == 2017 else 5)
    dps_w = [get_val(frame, "dps_common", y) for y in range(fy - win_len + 1, fy + 1)]
    paid_w = sum(1 for v in dps_w if v is not None and v > 0)
    cuts_w = sum(1 for a, b in zip(dps_w, dps_w[1:]) if a is not None and b is not None and a > 0 and b < a * 0.99)

    if fy == 2016:
        g_dps = cagr_avg([v for y in (2014,) if (v := get_val(frame, "dps_common", y)) is not None],
                         [v for y in (2016,) if (v := get_val(frame, "dps_common", y)) is not None], 2.0, min_len=1)
        g_eps = cagr_avg([v for y in (2014,) if (v := get_val(frame, "eps_reported", y)) is not None],
                         [v for y in (2016,) if (v := get_val(frame, "eps_reported", y)) is not None], 2.0, min_len=1)
    elif fy == 2017:
        g_dps = cagr_avg([v for y in (2014, 2015) if (v := get_val(frame, "dps_common", y)) is not None],
                         [v for y in (2016, 2017) if (v := get_val(frame, "dps_common", y)) is not None], 2.0, min_len=2)
        g_eps = cagr_avg([v for y in (2014, 2015) if (v := get_val(frame, "eps_reported", y)) is not None],
                         [v for y in (2016, 2017) if (v := get_val(frame, "eps_reported", y)) is not None], 2.0, min_len=2)
    elif fy <= 2019:
        e_rng = range(2013, 2016)
        l_rng = range(fy - 2, fy + 1)
        span = 3.0 if fy == 2018 else 3.5
        g_dps = cagr_avg([v for y in e_rng if (v := get_val(frame, "dps_common", y)) is not None],
                         [v for y in l_rng if (v := get_val(frame, "dps_common", y)) is not None], span)
        g_eps = cagr_avg([v for y in e_rng if (v := get_val(frame, "eps_reported", y)) is not None],
                         [v for y in l_rng if (v := get_val(frame, "eps_reported", y)) is not None], span)
    else:
        g_dps = cagr_avg([v for y in range(fy - 7, fy - 4) if (v := get_val(frame, "dps_common", y)) is not None],
                         [v for y in range(fy - 2, fy + 1) if (v := get_val(frame, "dps_common", y)) is not None], 5.0)
        g_eps = cagr_avg([v for y in range(fy - 7, fy - 4) if (v := get_val(frame, "eps_reported", y)) is not None],
                         [v for y in range(fy - 2, fy + 1) if (v := get_val(frame, "eps_reported", y)) is not None], 5.0)

    min_paid_fail = max(2, win_len - 1)
    v1 = "FAIL" if (paid_w < min_paid_fail or cuts_w >= 2) else ("PASS" if (paid_w == win_len and cuts_w == 0 and g_dps is not None and g_dps > 0) else "WARN")

    # --- 5번 총환원율 & 배당성향 ---
    d3 = [get_val(frame, "dividends_paid", y) if get_val(frame, "dividends_paid", y) is not None else get_val(frame, "div_total_reported", y) for y in range(fy - 2, fy + 1)]
    ni3 = [get_val(frame, "net_income_parent", y) if get_val(frame, "net_income_parent", y) is not None else get_val(frame, "ni_reported", y) for y in range(fy - 2, fy + 1)]
    tp3 = [get_val(frame, "treasury_purchase", y) or 0.0 for y in range(fy - 2, fy + 1)]
    canc3 = sum(get_val(frame, "ts_cancelled_common", y) or 0.0 for y in range(fy - 2, fy + 1))
    acq3 = sum(get_val(frame, "ts_acquired_common", y) or 0.0 for y in range(fy - 2, fy + 1))
    canc_ratio3 = min(1.0, canc3 / acq3) if acq3 > 0 else 0.0

    p3 = (sum(d3) / sum(ni3) * 100.0) if (all(v is not None for v in d3 + ni3) and sum(ni3) > 0) else None
    tr_3y = ((sum(d3) + sum(tp3)) / sum(ni3) * 100.0) if (all(v is not None for v in d3 + ni3) and sum(ni3) > 0) else None

    # --- 2번 EPS 성장 ---
    if g_eps is None or g_dps is None:
        v2 = "NA"
    elif g_eps < 0 and g_dps > 0:
        v2 = "FAIL"
    elif g_eps > 0 and (g_dps <= g_eps + 0.03 or (p3 is not None and p3 < 30.0)):
        v2 = "PASS"
    else:
        v2 = "WARN"

    # --- 3번 FCF / 금융 자본 ---
    if path == "general":
        k3 = [v for y in range(max(2014, fy - 4), fy + 1) if (v := get_val(frame, "fcf", y)) is not None]
        v3 = "FAIL" if sum(1 for v in k3 if v < 0) >= 2 else ("PASS" if (len(k3) >= 3 and sum(1 for v in k3 if v > 0) / len(k3) >= 0.75) else ("WARN" if len(k3) >= 2 else "NA"))
    else:
        eqs = [get_val(frame, "equity_parent", y) for y in range(max(2014, fy - 4), fy + 1)]
        eq_ch = [b - a for a, b in zip(eqs, eqs[1:]) if a is not None and b is not None]
        v3 = "FAIL" if sum(1 for v in eq_ch if v < 0) >= 2 else ("PASS" if (len(eq_ch) >= 2 and sum(1 for v in eq_ch if v > 0) / len(eq_ch) >= 0.75) else ("WARN" if eq_ch else "NA"))

    # --- 4번 자본효율 ---
    sps = []
    for y in range(max(2014, fy - 4), fy + 1):
        rf = RF_MAP.get(y, 0.025)
        coe = rf + 0.05
        if path != "general":
            roe = get_val(frame, "roe", y)
            if roe is not None:
                sps.append(roe - coe)
        else:
            roic = get_val(frame, "roic", y)
            if roic is not None:
                nd = get_val(frame, "net_debt", y)
                wacc = coe if (nd is None or nd <= 0) else (0.75 * coe + 0.25 * (rf + 0.015) * 0.78)
                sps.append(roic - wacc)
    avg5_sp = (sum(sps) / len(sps)) if sps else None
    p_thr = 0.03 if path == "general" else 0.01
    if len(sps) >= 3 and all(s < 0 for s in sps[-3:]):
        v4 = "FAIL"
    elif avg5_sp is not None:
        v4 = "PASS" if avg5_sp >= p_thr else "WARN"
    else:
        v4 = "NA"

    # --- 5번 판정 ---
    if tr_3y is None or p3 is None:
        v5 = "NA"
    elif tr_3y < 20.0 or p3 > 75.0 or (path != "general" and tr_3y > 100.0):
        v5 = "FAIL"
    elif tr_3y >= 30.0 and p3 <= 60.0 and (path == "general" or tr_3y <= 66.7):
        v5 = "PASS"
    else:
        v5 = "WARN"

    # --- 6번 자사주·희석 [버그 1 보정: 액면분할 split_factor_to_latest 반영!] ---
    s_now = get_val(frame, "outstanding_common", fy)
    sh_old_y = fy - 5
    s_old = get_val(frame, "outstanding_common", sh_old_y)
    if not s_old:
        avail_sh = sorted(int(yk) for yk, yv in frame.items() if int(yk) <= fy - 3 and (yv.get("outstanding_common") or 0) > 0)
        if avail_sh:
            sh_old_y = min(avail_sh, key=lambda yk: abs((fy - yk) - 5))
            s_old = get_val(frame, "outstanding_common", sh_old_y)
    if s_now and s_old and s_old > 0 and fy > sh_old_y:
        sf_now = get_val(frame, "split_factor_to_latest", fy) or 1.0
        sf_old = get_val(frame, "split_factor_to_latest", sh_old_y) or 1.0
        sh_chg = ((s_now * sf_now) / (s_old * sf_old)) ** (5.0 / (fy - sh_old_y)) - 1.0
    else:
        sh_chg = None
    canc5 = sum(get_val(frame, "ts_cancelled_common", y) or 0.0 for y in range(max(2015, fy - 4), fy + 1))
    acq5 = sum(get_val(frame, "ts_acquired_common", y) or 0.0 for y in range(max(2015, fy - 4), fy + 1))
    if sh_chg is None:
        v6 = "NA"
    elif sh_chg >= 0.03:
        v6 = "FAIL"
    elif (sh_chg < -0.01 and canc5 > 0) or (-0.01 <= sh_chg <= 0.01 and (acq5 == 0 or canc5 >= acq5)):
        v6 = "PASS"
    else:
        v6 = "WARN"

    # --- 7번 환원감당 & 8번 건전성 ---
    if path != "general":
        v7, v8 = "참고", "PASS"
    else:
        fcfs3 = [get_val(frame, "fcf", y) for y in range(fy - 2, fy + 1)]
        if any(v is None for v in d3 + fcfs3):
            v7 = "NA"
        elif sum(fcfs3) <= 0:
            v7 = "FAIL" if (sum(d3) + sum(tp3)) > 0 else "PASS"
        else:
            cov = (sum(d3) + sum(tp3)) / sum(fcfs3)
            v7 = "FAIL" if cov > 1.0 else ("WARN" if cov > 2 / 3 else "PASS")
        nd, nde, icr = get_val(frame, "net_debt", fy), get_val(frame, "net_debt_to_ebitda", fy), get_val(frame, "interest_coverage", fy)
        if nd is not None and nd <= 0:
            v8 = "PASS"
        elif (icr is not None and icr < 3.0) or (nde is not None and nde > 3.0):
            v8 = "FAIL"
        elif nde is not None and icr is not None:
            v8 = "PASS" if (nde < 2.0 and icr >= 5.0) else "WARN"
        else:
            v8 = "NA"

    counted = [v1, v2, v3, v4, v5, v6] + ([v7, v8] if path == "general" else [])
    n_fail, n_warn, n_na = counted.count("FAIL"), counted.count("WARN"), sum(1 for v in counted if v == "NA")
    pass_a = (n_fail == 0 and n_warn <= 2 and n_na <= 2)
    if (not include_non_pass) and (not pass_a):
        return None

    # --- PIT 시총 & 절대/상대 밸류에이션 ---
    dps_y_adj = act_dps.get(fy)
    dps_y_rep = get_val(frame, "dps_common", fy)
    adj_ratio = (dps_y_adj / dps_y_rep) if (dps_y_adj and dps_y_rep and dps_y_rep > 0) else 1.0
    eps_y = get_val(frame, "eps_reported", fy)
    eq_y = get_val(frame, "equity_parent", fy)

    # 발행주식수: FY2016~2017은 해당 연도 결측 시 2015년 주식수 폴백 (분할계수 일치 보정)
    sh_yr = fy
    sh_tot = (get_val(frame, "outstanding_common", fy) or 0) + (get_val(frame, "outstanding_pref", fy) or 0)
    if sh_tot <= 0 and fy < 2018:
        sh_yr = 2015
        sh_tot = (get_val(frame, "outstanding_common", 2015) or 0) + (get_val(frame, "outstanding_pref", 2015) or 0)

    split_fac = get_val(frame, "split_factor_to_latest", sh_yr)
    if split_fac is None or split_fac <= 0:
        split_fac = (1.0 / adj_ratio) if (adj_ratio and adj_ratio > 0) else 1.0

    pit_mcap_eok = (px_now * split_fac * sh_tot) / 1e8 if sh_tot > 0 else None
    if FINAL_CONFIG["PIT_MIN_MCAP_EOK"] > 0 and (pit_mcap_eok is None or pit_mcap_eok < FINAL_CONFIG["PIT_MIN_MCAP_EOK"]):
        return None

    bps_y = (eq_y / sh_tot) if (eq_y and sh_tot > 0) else None
    bps_adj_ratio = (1.0 / split_fac) if (sh_yr != fy and split_fac > 0) else adj_ratio

    c_per = px_now / (eps_y * adj_ratio) if (eps_y and eps_y > 0) else None
    c_pbr = px_now / (bps_y * bps_adj_ratio) if (bps_y and bps_y > 0) else None
    c_yld = dps_y_adj / px_now if (dps_y_adj and dps_y_adj > 0) else None

    h_per, h_pbr, h_yld = [], [], []
    for y in range(max(2014, fy - 7), fy + 1):
        sub_y = px_df.loc[f"{y}-01-01":f"{y}-12-31"]
        if sub_y.empty:
            continue
        py = float(sub_y["close"].iloc[-1])
        ey = get_val(frame, "eps_reported", y)
        eq = get_val(frame, "equity_parent", y)
        st = (get_val(frame, "outstanding_common", y) or 0) + (get_val(frame, "outstanding_pref", y) or 0)
        if st <= 0 and y < 2018:
            st = (get_val(frame, "outstanding_common", 2015) or 0) + (get_val(frame, "outstanding_pref", 2015) or 0)
        by = (eq / st) if (eq and st > 0) else None
        dy_a = act_dps.get(y)
        dy_r = get_val(frame, "dps_common", y)
        ar = (dy_a / dy_r) if (dy_a and dy_r and dy_r > 0) else adj_ratio
        if ey and ey > 0:
            h_per.append(py / (ey * ar))
        if by and by > 0:
            h_pbr.append(py / (by * ar))
        if dy_a and dy_a > 0:
            h_yld.append(dy_a / py)

    devs_raw, devs_cap = [], []
    cap_lim = FINAL_CONFIG["VAL_CAP"]
    min_hv = 2 if fy <= 2018 else 3
    for cv, hv, hc in ((c_per, h_per, False), (c_pbr, h_pbr, False), (c_yld, h_yld, True)):
        if cv is not None and len(hv) >= min_hv:
            med = statistics.median(hv)
            if med > 0:
                d = (cv / med - 1.0) if hc else -(cv / med - 1.0)
                devs_raw.append(d)
                devs_cap.append(max(-cap_lim, min(cap_lim, d)))
    val_raw = sum(devs_raw) / len(devs_raw) if len(devs_raw) >= 2 else None
    val_cap = sum(devs_cap) / len(devs_cap) if len(devs_cap) >= 2 else None
    if (not include_non_pass) and val_cap is None:
        return None

    closes = hist["close"]
    dret = closes.tail(252).pct_change().dropna()
    ann_vol = float(dret.std() * np.sqrt(252)) if len(dret) >= 30 else 0.30
    turnover_60d_eok = float((hist["close"] * hist["volume"]).tail(60).mean()) / 1e8

    # [버그 4 보정] 진성 주주환원수익률 = 연평균 (현금배당 + 소각한 자사주 몫) / 매수시점 PIT 시총
    pit_mcap_won = pit_mcap_eok * 1e8
    if all(v is not None for v in d3) and pit_mcap_won > 0:
        true_sh_yield = ((sum(d3) + sum(tp3) * canc_ratio3) / 3.0) / pit_mcap_won
    else:
        true_sh_yield = c_yld

    # 운용규칙 4절 A 매도 사유 (FAIL / 2년 연속 삭감 / 배당 중단)
    dps_m2, dps_m1, dps_0 = get_val(frame, "dps_common", fy - 2), get_val(frame, "dps_common", fy - 1), get_val(frame, "dps_common", fy)
    consec_2_cuts = bool(
        dps_m2 is not None and dps_m1 is not None and dps_0 is not None
        and dps_m2 > 0 and dps_m1 < dps_m2 * 0.99 and dps_0 < dps_m1 * 0.99
    )
    sell_trigger = (n_fail > 0) or consec_2_cuts or (dps_0 is not None and dps_0 <= 0)

    abs_over = (
        (c_per is not None and c_per > FINAL_CONFIG["MAX_ABS_PER"])
        or (path == "general" and c_pbr is not None and c_pbr > FINAL_CONFIG["MAX_ABS_PBR"])
    )
    abs_ok = (not abs_over) and ((c_per is not None and c_pbr is not None) if path == "general" else True)

    return {
        "fy": fy,
        "rebal_dt": rebal_dt.strftime("%Y-%m-%d"),
        "ticker": t,
        "name": comp["name"],
        "market": comp.get("market", "KOSPI"),
        "path": path,
        "is_fin": path != "general",
        "pass_a": pass_a,
        "sell_trigger": sell_trigger,
        "pit_mcap_eok": pit_mcap_eok,
        "ann_vol": ann_vol,
        "turnover_60d_eok": turnover_60d_eok,
        "per": c_per,
        "pbr": c_pbr,
        "yield_fy": c_yld,
        "tr_3y": tr_3y,
        "true_sh_yield": true_sh_yield,
        "abs_ok": abs_ok,
        "f_val_raw": val_raw,
        "f_val": val_cap,
        "f_dps_g": g_dps if g_dps is not None else 0.0,
    }

REBAL_DATES = {}
for fy in range(FINAL_CONFIG["EXT_START_FY"], FINAL_CONFIG["END_FY"] + 1):
    sub = CALENDAR[CALENDAR >= pd.Timestamp(f"{fy + 1}-04-01")]
    if len(sub) > 0:
        REBAL_DATES[fy] = sub[0]

PANEL_ROWS, SELL_MAP = [], {}
for fy in sorted(REBAL_DATES.keys()):
    dt = REBAL_DATES[fy]
    s_flags = {}
    for comp in COMPANIES:
        rec = extract_company_year_clean(comp, fy, dt, include_non_pass=True)
        if rec is None:
            continue
        s_flags[rec["ticker"]] = rec["sell_trigger"]
        if rec["pass_a"] and rec["f_val"] is not None:
            PANEL_ROWS.append(rec)
    SELL_MAP[dt] = s_flags

PANEL_DF = pd.DataFrame(PANEL_ROWS)
print(f"[클린 PIT 패널 완료] 총 {len(PANEL_DF)}개 기업-연도 (FY{FINAL_CONFIG['EXT_START_FY']}~FY{FINAL_CONFIG['END_FY']}) | 연도별 후보 수:")
print(PANEL_DF.groupby("rebal_dt")["ticker"].count().to_string())

In [ ]:
# ==============================================================================
# 5. [검증 ①] 바구니 A 단독 비교 (공식 7.7년 2019-04~ & 확장 9.7년 2017-04~, 거치식 + 적립식)
# ==============================================================================
_PX_PANEL = pd.DataFrame({t: df["close"] for t, df in PRICE_MAP.items()}).reindex(CALENDAR).ffill()
_PX_ARR = np.nan_to_num(_PX_PANEL.to_numpy(dtype=np.float64), nan=0.0)
_TK_COL = {t: idx for idx, t in enumerate(_PX_PANEL.columns)}
_CAL_POS = {dt: idx for idx, dt in enumerate(CALENDAR)}
_ACT_DPS_MAP = {c["ticker"]: {int(k): float(v) for k, v in (c.get("act_dps") or {}).items() if v is not None} for c in COMPANIES}
_APRIL_DAYS = {y: CALENDAR[(CALENDAR.year == y) & (CALENDAR.month == 4)][0] for y in sorted(set(CALENDAR.year)) if len(CALENDAR[(CALENDAR.year == y) & (CALENDAR.month == 4)]) > 0}
_APRIL_SET = set(_APRIL_DAYS.values())

def pct_rank(s: pd.Series, higher_better: bool = True) -> pd.Series:
    return (s if higher_better else -s).rank(pct=True, method="average").fillna(0.5)

def score_dna(g: pd.DataFrame, use_true_sh_yield: bool = True) -> pd.Series:
    r_vol = pct_rank(pd.to_numeric(g["ann_vol"], errors="coerce"), higher_better=False)
    per_s = pd.to_numeric(g["per"], errors="coerce")
    r_per = pct_rank(per_s.where(per_s > 0), higher_better=False)
    r_val_cap = pct_rank(pd.to_numeric(g["f_val"], errors="coerce").fillna(0.0), higher_better=True)
    pbr_s = pd.to_numeric(g["pbr"], errors="coerce")
    nonfin_pbr = pbr_s.where((~g["is_fin"]) & (pbr_s > 0))
    r_pbr = pct_rank(nonfin_pbr, higher_better=False)
    r_abs_val = np.where(g["is_fin"], 0.5 * r_per + 0.5 * r_val_cap, 0.5 * r_per + 0.5 * r_pbr)
    r_yld = pct_rank(pd.to_numeric(g["yield_fy"], errors="coerce"), higher_better=True)
    tr_col = "true_sh_yield" if use_true_sh_yield else "tr_3y"
    r_tr = pct_rank(pd.to_numeric(g[tr_col], errors="coerce").fillna(0.0), higher_better=True)
    return 0.30 * r_vol + 0.30 * pd.Series(r_abs_val, index=g.index) + 0.20 * r_yld + 0.20 * r_tr

def build_strategy_weights(panel: pd.DataFrame, mode: str, start_fy: int, top_n: int = 10, max_fin_ratio: float | None = None):
    sub = panel[panel["fy"] >= start_fy]
    w_map, ranked_map, picks_log, pass_cnt = {}, {}, {}, {}
    min_liq = FINAL_CONFIG["MIN_LIQ_EOK"]

    for dt_str in sorted(sub["rebal_dt"].unique()):
        dt = pd.Timestamp(dt_str)
        g_all = sub[sub["rebal_dt"] == dt_str].copy()
        g = g_all[g_all["turnover_60d_eok"] >= min_liq].copy()
        if len(g) < top_n:
            g = g_all.copy()

        if mode == "A_clean":
            g["base_sc"] = pct_rank(g["f_val_raw"], True) + pct_rank(g["f_dps_g"], True)
            gate = pd.Series(True, index=g.index)
        elif mode == "DNA_tr3y_only":
            g["base_sc"] = score_dna(g, use_true_sh_yield=False)
            gate = pd.Series(True, index=g.index)
        elif mode == "DNA_fixed35":
            g["base_sc"] = score_dna(g, use_true_sh_yield=False)
            gate = g["abs_ok"] & (g["ann_vol"] <= FINAL_CONFIG["FIXED_VOL_CAP"])
        elif mode == "DNA_final_rel70":
            g["base_sc"] = score_dna(g, use_true_sh_yield=True)
            vcut = float(g["ann_vol"].quantile(FINAL_CONFIG["REL_VOL_QUANTILE"]))
            gate = g["abs_ok"] & (g["ann_vol"] <= vcut)
        elif mode == "DNA_final_nogate":
            g["base_sc"] = score_dna(g, use_true_sh_yield=True)
            gate = pd.Series(True, index=g.index)
        elif mode == "EW_pass":
            w_map[dt] = {t: 1.0 / len(g) for t in g["ticker"]}
            ranked_map[dt] = g["ticker"].tolist()
            pass_cnt[dt_str[:4]] = f"{len(g)}/{len(g)}"
            continue
        else:
            raise ValueError(mode)

        pass_cnt[dt_str[:4]] = f"{int(gate.sum())}/{len(g)}"
        g["sc"] = gate.astype(float) * 10.0 + g["base_sc"]
        g_sorted = g.sort_values("sc", ascending=False)

        if max_fin_ratio is not None:
            max_fin = max(1, int(round(top_n * max_fin_ratio)))
            chosen, f_cnt = [], 0
            for _, r in g_sorted.iterrows():
                if len(chosen) >= top_n:
                    break
                if r["is_fin"] and f_cnt >= max_fin:
                    continue
                chosen.append(r)
                if r["is_fin"]:
                    f_cnt += 1
            if len(chosen) < top_n:
                used = {r["ticker"] for r in chosen}
                for _, r in g_sorted.iterrows():
                    if r["ticker"] not in used:
                        chosen.append(r)
                        if len(chosen) >= top_n:
                            break
            picks = pd.DataFrame(chosen)
            rk = picks["ticker"].tolist() + [t for t in g_sorted["ticker"].tolist() if t not in set(picks["ticker"])]
        else:
            picks = g_sorted.head(top_n)
            rk = g_sorted["ticker"].tolist()

        w_map[dt] = {t: 1.0 / len(picks) for t in picks["ticker"]}
        ranked_map[dt] = rk
        picks_log[dt_str] = list(zip(picks["ticker"], picks["name"], picks["is_fin"]))

    return w_map, ranked_map, picks_log, pass_cnt

def run_lumpsum_clean(weights_by_date: dict) -> dict:
    """[버그 2 보정] 리밸런싱 여부와 무관하게 매년 4월 첫 영업일에 세후 배당금 정상 수취·재투자."""
    rebal_set = sorted(weights_by_date.keys())
    start_pos = _CAL_POS[rebal_set[0]]
    cal = CALENDAR[start_pos:]
    nav_arr = np.empty(len(cal), dtype=np.float64)
    fee, div_tax = FINAL_CONFIG["FEE_RATE"], FINAL_CONFIG["DIV_TAX_RATE"]
    shares, cash, div_paid_years, to_log = {}, float(FINAL_CONFIG["INITIAL_CAPITAL"]), set(), []

    for i, dt in enumerate(cal):
        px_row = _PX_ARR[start_pos + i]
        if shares and (dt in _APRIL_SET) and (dt.year not in div_paid_years):
            gross = sum(sh * _ACT_DPS_MAP.get(tk, {}).get(dt.year - 1, 0.0) for tk, sh in shares.items())
            cash += gross * (1.0 - div_tax)
            div_paid_years.add(dt.year)
            if (dt not in weights_by_date) and cash > 0:
                sv = sum(sh * px_row[_TK_COL[tk]] for tk, sh in shares.items() if tk in _TK_COL)
                if sv > 0:
                    inv = max(0.0, cash * (1.0 - fee))
                    for tk in list(shares.keys()):
                        px = px_row[_TK_COL[tk]]
                        if px > 0:
                            shares[tk] += (inv * (shares[tk] * px / sv)) / px
                    cash = 0.0

        if dt in weights_by_date:
            sv = sum(sh * px_row[_TK_COL[tk]] for tk, sh in shares.items() if tk in _TK_COL)
            tv = sv + cash
            valid_w = {tk: w for tk, w in weights_by_date[dt].items() if tk in _TK_COL and px_row[_TK_COL[tk]] > 0}
            ws = sum(valid_w.values())
            if ws > 0:
                valid_w = {tk: w / ws for tk, w in valid_w.items()}
                traded = sum(abs(tv * valid_w.get(tk, 0.0) - shares.get(tk, 0.0) * (px_row[_TK_COL[tk]] if tk in _TK_COL else 0.0)) for tk in (set(shares) | set(valid_w)))
                to_log.append((dt.strftime("%Y-%m-%d"), traded / (2.0 * tv) if tv > 0 else 0.0))
                inv = max(0.0, tv - traded * fee)
                shares = {tk: (inv * w) / px_row[_TK_COL[tk]] for tk, w in valid_w.items()}
                cash = 0.0

        nav_arr[i] = cash + sum(sh * px_row[_TK_COL[tk]] for tk, sh in shares.items() if tk in _TK_COL)

    return {"nav": pd.Series(nav_arr, index=cal), "turnover": to_log}

def calc_xirr(cfs: list[tuple[pd.Timestamp, float]]) -> float:
    t0 = cfs[0][0]
    cy = [((d - t0).days / 365.25, a) for d, a in cfs]
    lo, hi = -0.90, 5.0
    for _ in range(100):
        mid = 0.5 * (lo + hi)
        if sum(a / ((1.0 + mid) ** y) for y, a in cy) > 0:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)

def run_dca_longhold_clean(ranked_by_date: dict, top_n: int = 10) -> dict:
    """매월 100만 원 적립 + 운용규칙 4절 A(FAIL/2연속삭감 외 매도 금지) 장기보유."""
    rebal_dates = sorted(ranked_by_date.keys())
    start_pos = _CAL_POS[rebal_dates[0]]
    cal = CALENDAR[start_pos:]
    ym = cal.to_period("M")
    first_m = set()
    seen = set()
    for dt, m in zip(cal, ym):
        if m not in seen:
            first_m.add(dt); seen.add(m)

    fee, div_tax, m_cash = FINAL_CONFIG["FEE_RATE"], FINAL_CONFIG["DIV_TAX_RATE"], FINAL_CONFIG["MONTHLY_DCA_A_KRW"]
    shares, cash, div_paid_years, cur_rk, cfs = {}, 0.0, set(), [], []
    total_in = total_div = 0.0
    sells = 0
    for i, dt in enumerate(cal):
        px_row = _PX_ARR[start_pos + i]
        if shares and (dt in _APRIL_SET) and (dt.year not in div_paid_years):
            gross = sum(sh * _ACT_DPS_MAP.get(tk, {}).get(dt.year - 1, 0.0) for tk, sh in shares.items())
            net = gross * (1.0 - div_tax)
            cash += net; total_div += net; div_paid_years.add(dt.year)
        if dt in ranked_by_date:
            cur_rk = [tk for tk in ranked_by_date[dt] if tk in _TK_COL and px_row[_TK_COL[tk]] > 0]
            sflags = SELL_MAP.get(dt, {})
            for tk in list(shares.keys()):
                if sflags.get(tk, False):
                    cash += shares.pop(tk) * px_row[_TK_COL[tk]] * (1.0 - fee)
                    sells += 1
        if dt in first_m:
            cash += m_cash; total_in += m_cash; cfs.append((dt, -m_cash))
        if cash > 0 and cur_rk:
            tgts = [tk for tk in cur_rk[:top_n] if tk in _TK_COL and px_row[_TK_COL[tk]] > 0]
            if tgts:
                per = cash * (1.0 - fee) / len(tgts)
                for tk in tgts:
                    shares[tk] = shares.get(tk, 0.0) + per / px_row[_TK_COL[tk]]
                cash = 0.0
    fv = cash + sum(sh * _PX_ARR[start_pos + len(cal) - 1, _TK_COL[tk]] for tk, sh in shares.items() if tk in _TK_COL)
    cfs.append((cal[-1], fv))
    return {"final_val": fv, "total_in": total_in, "xirr": calc_xirr(cfs), "total_div": total_div, "sells": sells}

def calc_perf(nav: pd.Series, to_log=None, start_dt=None, end_dt=None) -> dict:
    s = nav
    if start_dt is not None:
        s = s.loc[s.index >= pd.Timestamp(start_dt)]
    if end_dt is not None:
        s = s.loc[s.index <= pd.Timestamp(end_dt)]
    yrs = (s.index[-1] - s.index[0]).days / 365.25
    tot = float(s.iloc[-1] / s.iloc[0] - 1.0)
    cagr = float((s.iloc[-1] / s.iloc[0]) ** (1.0 / yrs) - 1.0) if yrs > 0 else 0.0
    mdd = float((s / s.cummax() - 1.0).min())
    vol = float(s.pct_change().dropna().std() * np.sqrt(252))
    rf = float(np.mean([RF_MAP.get(y, 0.028) for y in sorted(set(s.index.year))]))
    sharpe = (cagr - rf) / vol if vol > 0 else 0.0
    to = float(np.mean([x[1] for x in to_log[1:]])) if (to_log and len(to_log) > 1) else np.nan
    return {"tot": tot, "cagr": cagr, "mdd": mdd, "vol": vol, "sharpe": sharpe, "to": to, "yrs": yrs}

STRAT_SPECS = [
    ("현행 전략 A (클린 기준선)",               "A_clean",          None),
    ("생존 DNA 점수만 (기존 tr_3y)",            "DNA_tr3y_only",    None),
    ("생존 DNA + 고정 35% 게이트 (기존)",        "DNA_fixed35",      None),
    ("최종안 ①: 진성환원 + 상대70% 게이트",      "DNA_final_rel70",  None),
    ("최종안 ②: ① + 금융 ≤ 50% 상한 (권장)",    "DNA_final_rel70",  FINAL_CONFIG["MAX_FIN_RATIO_IN_A"]),
    ("최종안 ③: 진성환원 점수만 + 금융 ≤ 50%",   "DNA_final_nogate", FINAL_CONFIG["MAX_FIN_RATIO_IN_A"]),
    ("원칙부합 전체 동일가중 (EW)",             "EW_pass",          None),
]

A_NAVS_9Y, A_NAVS_7Y = {}, {}
A_TABLE_9Y, A_TABLE_7Y = [], []
LAST_PICKS_FINAL = {}

for horizon_label, sfy, out_tbl, out_navs in [
    ("확장 9.7년 PIT (2017-04~)", FINAL_CONFIG["EXT_START_FY"], A_TABLE_9Y, A_NAVS_9Y),
    ("공식 7.7년 PIT (2019-04~)", FINAL_CONFIG["OFFICIAL_START_FY"], A_TABLE_7Y, A_NAVS_7Y),
]:
    st_dt = REBAL_DATES[sfy]
    k_sub = KOSPI_DF.loc[st_dt:, "close"]
    k_pr = k_sub / k_sub.iloc[0] * FINAL_CONFIG["INITIAL_CAPITAL"]
    k_tr = k_pr * ((1.0 + FINAL_CONFIG["KOSPI_EST_DIV_YIELD"]) ** ((k_sub.index - k_sub.index[0]).days / 365.25))
    out_navs["KOSPI_TR"] = k_tr / k_tr.iloc[0]

    for label, mode, fin_cap in STRAT_SPECS:
        w, rk, plog, pcnt = build_strategy_weights(PANEL_DF, mode, sfy, FINAL_CONFIG["TOP_N"], fin_cap)
        res = run_lumpsum_clean(w)
        out_navs[label] = res["nav"] / res["nav"].iloc[0]
        mf = calc_perf(res["nav"], res["turnover"])
        m_pre = calc_perf(res["nav"], start_dt=st_dt, end_dt="2022-04-01")
        m_post = calc_perf(res["nav"], start_dt="2022-04-01")
        dca = run_dca_longhold_clean(rk, FINAL_CONFIG["TOP_N"]) if mode != "EW_pass" else None
        out_tbl.append({
            "전략": label,
            "거치 CAGR": f"{mf['cagr']:+.2%}",
            "MDD": f"{mf['mdd']:.1%}",
            "샤프": f"{mf['sharpe']:.2f}",
            "전반기(~22.04)": f"{m_pre['cagr']:+.2%}",
            "후반기(22.04~)": f"{m_post['cagr']:+.2%}",
            "적립 XIRR": "-" if dca is None else f"{dca['xirr']:+.2%}",
            "적립 누적배당": "-" if dca is None else f"{dca['total_div']/1e6:.1f}백만",
        })
        if sfy == FINAL_CONFIG["EXT_START_FY"] and "최종안 ②" in label:
            LAST_PICKS_FINAL = plog

    mk = calc_perf(k_tr)
    mk1 = calc_perf(k_tr, start_dt=st_dt, end_dt="2022-04-01")
    mk2 = calc_perf(k_tr, start_dt="2022-04-01")
    out_tbl.append({
        "전략": "KOSPI TR 추정(+1.8%/년)",
        "거치 CAGR": f"{mk['cagr']:+.2%}",
        "MDD": f"{mk['mdd']:.1%}",
        "샤프": f"{mk['sharpe']:.2f}",
        "전반기(~22.04)": f"{mk1['cagr']:+.2%}",
        "후반기(22.04~)": f"{mk2['cagr']:+.2%}",
        "적립 XIRR": "-",
        "적립 누적배당": "-",
    })

DF_A_9Y = pd.DataFrame(A_TABLE_9Y)
DF_A_7Y = pd.DataFrame(A_TABLE_7Y)
print("■ [검증 ①-A] 바구니 A 단독 — 확장 9.7년 PIT (2017-04 ~ 최신, 2018 하락장 포함)")
display(DF_A_9Y)
print("\n■ [검증 ①-B] 바구니 A 단독 — 공식 7.7년 PIT (2019-04 ~ 최신)")
display(DF_A_7Y)

In [ ]:
# ==============================================================================
# 6. [검증 ②] A+B+C 통합 포트폴리오 (50/20/30 vs 40/15/45, 원화 환노출, 적립식)
# ==============================================================================
def fetch_yf_adj(sym: str, range_str: str = "20y") -> pd.Series:
    r = requests.get(
        f"https://query1.finance.yahoo.com/v8/finance/chart/{sym}",
        params={"range": range_str, "interval": "1d"},
        headers={"User-Agent": "Mozilla/5.0 (dividend-final-bt)"},
        timeout=20,
    )
    r.raise_for_status()
    res = r.json()["chart"]["result"][0]
    ts = res["timestamp"]
    adj = res["indicators"].get("adjclose", [{}])[0].get("adjclose")
    vals = adj if adj is not None else res["indicators"]["quote"][0]["close"]
    dates = [pd.Timestamp(datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m-%d")) for t in ts]
    s = pd.Series(vals, index=dates, dtype=float).dropna()
    return s[~s.index.duplicated(keep="last")].sort_index()

print("[외부 시세 수집] Yahoo Finance 20년 일봉 (SPY · SCHD · SOXX · KRW=X · 005935.KS · ^KS11)...")
YF_SPY = fetch_yf_adj("SPY")
YF_SCHD = fetch_yf_adj("SCHD")
YF_SOXX = fetch_yf_adj("SOXX")
YF_FX = fetch_yf_adj("KRW=X")
YF_SEC_PREF = fetch_yf_adj("005935.KS")
YF_KS11 = fetch_yf_adj("^KS11")
print(f"  수집 완료: SCHD {YF_SCHD.index[0].date()} ~ {YF_SCHD.index[-1].date()} ({len(YF_SCHD):,}일)")

def build_bc_navs(target_cal: pd.DatetimeIndex) -> tuple[pd.Series, pd.Series]:
    fx_a = YF_FX.reindex(target_cal, method="ffill").bfill()
    yrs = (target_cal - target_cal[0]).days / 365.25
    spy_k = YF_SPY.reindex(target_cal, method="ffill").bfill() * fx_a
    schd_k = YF_SCHD.reindex(target_cal, method="ffill").bfill() * fx_a
    soxx_k = YF_SOXX.reindex(target_cal, method="ffill").bfill() * fx_a
    sec_k = YF_SEC_PREF.reindex(target_cal, method="ffill").bfill()

    # 보수적 ETF 보수 + 미국 배당원천징수(15%) 드래그 차감
    spy_net = (spy_k / spy_k.iloc[0]) * ((1.0 - 0.0030) ** yrs)
    schd_net = (schd_k / schd_k.iloc[0]) * ((1.0 - 0.0060) ** yrs)
    soxx_net = (soxx_k / soxx_k.iloc[0]) * ((1.0 - 0.0050) ** yrs)
    sec_net = (sec_k / sec_k.iloc[0]) * ((1.0 - 0.0040) ** yrs)  # 005935.KS adjclose 배당세 15.4% 감안

    def blend_annual(s1: pd.Series, w1: float, s2: pd.Series, w2: float) -> pd.Series:
        nav = np.empty(len(target_cal), dtype=float)
        u1, u2, last_y = w1 / s1.iloc[0], w2 / s2.iloc[0], target_cal[0].year
        for i, dt in enumerate(target_cal):
            if dt.year != last_y and dt.month == 4:
                tot = u1 * s1.iloc[i] + u2 * s2.iloc[i]
                u1, u2 = (tot * w1) / s1.iloc[i], (tot * w2) / s2.iloc[i]
                last_y = dt.year
            nav[i] = u1 * s1.iloc[i] + u2 * s2.iloc[i]
        return pd.Series(nav, index=target_cal)

    return blend_annual(sec_net, 0.50, soxx_net, 0.50), blend_annual(spy_net, 2.0 / 3.0, schd_net, 1.0 / 3.0)

def sim_3basket_lumpsum(na: pd.Series, nb: pd.Series, nc: pd.Series, targets: tuple[float, float, float]) -> pd.Series:
    wa, wb, wc = targets
    idx = na.index.intersection(nb.index).intersection(nc.index)
    sa, sb, sc = na.loc[idx], nb.loc[idx], nc.loc[idx]
    ua, ub, uc, last_y = wa / sa.iloc[0], wb / sb.iloc[0], wc / sc.iloc[0], idx[0].year
    fee = FINAL_CONFIG["PORT_FEE_RATE"]
    nav = np.empty(len(idx), dtype=float)
    for i, dt in enumerate(idx):
        va, vb, vc = ua * sa.iloc[i], ub * sb.iloc[i], uc * sc.iloc[i]
        tot = va + vb + vc
        if dt.year != last_y and dt.month == 4:
            if abs(va / tot - wa) > 0.05 or abs(vb / tot - wb) > 0.05 or abs(vc / tot - wc) > 0.05:
                traded = abs(tot * wa - va) + abs(tot * wb - vb) + abs(tot * wc - vc)
                tn = max(0.0, tot - traded * fee)
                ua, ub, uc = (tn * wa) / sa.iloc[i], (tn * wb) / sb.iloc[i], (tn * wc) / sc.iloc[i]
                tot = tn
            last_y = dt.year
        nav[i] = tot
    return pd.Series(nav, index=idx)

def sim_3basket_dca(na: pd.Series, nb: pd.Series, nc: pd.Series, targets: tuple[float, float, float]) -> dict:
    """운용규칙 5절 그대로: 매도 0건, 매월 200만 원을 허용폭·목표 미달 바구니에 집중 투입."""
    wa, wb, wc = targets
    idx = na.index.intersection(nb.index).intersection(nc.index)
    sa, sb, sc = na.loc[idx], nb.loc[idx], nc.loc[idx]
    ym = idx.to_period("M")
    first_m, seen = set(), set()
    for dt, m in zip(idx, ym):
        if m not in seen:
            first_m.add(dt); seen.add(m)
    m_cash, fee = FINAL_CONFIG["MONTHLY_DCA_PORT_KRW"], FINAL_CONFIG["PORT_FEE_RATE"]
    ua = ub = uc = total_in = 0.0
    cfs = []
    for i, dt in enumerate(idx):
        if dt in first_m:
            va, vb, vc = ua * sa.iloc[i], ub * sb.iloc[i], uc * sc.iloc[i]
            tot = va + vb + vc
            net_c = m_cash * (1.0 - fee)
            total_in += m_cash; cfs.append((dt, -m_cash))
            if tot <= 0:
                ua += (net_c * wa) / sa.iloc[i]; ub += (net_c * wb) / sb.iloc[i]; uc += (net_c * wc) / sc.iloc[i]
            else:
                cw = [va / tot, vb / tot, vc / tot]
                tgts = [wa, wb, wc]
                below = [k for k in range(3) if cw[k] < tgts[k] - 0.05]
                pick = max(below, key=lambda k: tgts[k] - cw[k]) if below else max(range(3), key=lambda k: tgts[k] - cw[k])
                if pick == 0: ua += net_c / sa.iloc[i]
                elif pick == 1: ub += net_c / sb.iloc[i]
                else: uc += net_c / sc.iloc[i]
    fv = ua * sa.iloc[-1] + ub * sb.iloc[-1] + uc * sc.iloc[-1]
    cfs.append((idx[-1], fv))
    return {"final_val": fv, "total_in": total_in, "xirr": calc_xirr(cfs), "mult": fv / total_in}

# 확장 9.7년 (2017-04 ~ 최신) 통합 포트폴리오 실행
CAL_9Y = A_NAVS_9Y["KOSPI_TR"].index
NAV_B_9Y, NAV_C_9Y = build_bc_navs(CAL_9Y)
A_FINAL_9Y = A_NAVS_9Y["최종안 ②: ① + 금융 ≤ 50% 상한 (권장)"]
A_ORIG_9Y = A_NAVS_9Y["현행 전략 A (클린 기준선)"]
K_TR_9Y = A_NAVS_9Y["KOSPI_TR"]

PORT_SPECS_9Y = [
    ("현행 50/20/30 (A=최종안 ② 권장)",      A_FINAL_9Y, (0.50, 0.20, 0.30)),
    ("권고 40/15/45 (A=최종안 ② 권장)",      A_FINAL_9Y, (0.40, 0.15, 0.45)),
    ("현행 50/20/30 (A=현행 전략 A)",        A_ORIG_9Y,  (0.50, 0.20, 0.30)),
    ("권고 40/15/45 (A=현행 전략 A)",        A_ORIG_9Y,  (0.40, 0.15, 0.45)),
    ("패시브 50/20/30 (A=KOSPI TR)",        K_TR_9Y,    (0.50, 0.20, 0.30)),
    ("패시브 40/15/45 (A=KOSPI TR)",        K_TR_9Y,    (0.40, 0.15, 0.45)),
    ("국내 100% (KOSPI TR)",               K_TR_9Y,    (1.00, 0.00, 0.00)),
    ("미국 100% (S&P500 2 : 배당다우 1)",   K_TR_9Y,    (0.00, 0.00, 1.00)),
]

PORT_NAVS_9Y, PORT_ROWS_9Y = {}, []
for nm, na, wt in PORT_SPECS_9Y:
    ls = sim_3basket_lumpsum(na, NAV_B_9Y, NAV_C_9Y, wt)
    PORT_NAVS_9Y[nm] = ls
    mf = calc_perf(ls)
    m1 = calc_perf(ls, start_dt=CAL_9Y[0], end_dt="2022-04-01")
    m2 = calc_perf(ls, start_dt="2022-04-01")
    dca = sim_3basket_dca(na, NAV_B_9Y, NAV_C_9Y, wt)
    PORT_ROWS_9Y.append({
        "통합 포트폴리오 (2017-04 ~ 최신, 9.7년)": nm,
        "거치 CAGR": f"{mf['cagr']:+.2%}",
        "MDD": f"{mf['mdd']:.1%}",
        "변동성": f"{mf['vol']:.1%}",
        "샤프": f"{mf['sharpe']:.2f}",
        "전반기(~22.04)": f"{m1['cagr']:+.2%}",
        "후반기(22.04~)": f"{m2['cagr']:+.2%}",
        "월200만 적립 XIRR": f"{dca['xirr']:+.2%}",
        "적립 최종평가": f"{dca['final_val']/1e8:.2f}억",
    })

DF_PORT_9Y = pd.DataFrame(PORT_ROWS_9Y)
CORR_9Y = pd.DataFrame({"A_최종권장": A_FINAL_9Y, "B_반도체AI": NAV_B_9Y, "C_미국지수": NAV_C_9Y, "KOSPI_TR": K_TR_9Y}).pct_change().dropna().corr()

print("■ [검증 ②] A+B+C 통합 포트폴리오 (확장 9.7년 PIT, 2017-04 ~ 최신)")
display(DF_PORT_9Y)
print("\n■ 바구니 간 일별 수익률 상관계수 (9.7년)")
display(CORR_9Y.round(2))

In [ ]:
# ==============================================================================
# 7. [검증 ③] 초장기 14.7년 추정 백테스트 (2012-04 ~ 최신, SCHD 상장 이후 전 구간)
# ==============================================================================
# ※ 정직한 한계: 2012~2016은 OpenDART XBRL 이전이라 재무 3~8번 미반영 & 상폐 생존편향 존재.
#   따라서 ① 보수적 추정(2012~2017 A=KOSPI TR -> 2017~ A=정밀 PIT, 생존편향 추가 0%)
#   ② Yahoo 배당·저변동 프록시(2012~2017 대형 배당주 저변동·고배당 -> 2017~ 정밀 PIT, 참고용) 두 가지를 분리 비교합니다!

DF_PORT_15Y = pd.DataFrame()
PORT_NAVS_15Y = {}

if FINAL_CONFIG["ENABLE_15Y_ESTIMATE"]:
    CAL_15Y = YF_KS11.loc["2012-04-01":].index
    yrs_15 = (CAL_15Y - CAL_15Y[0]).days / 365.25
    K_TR_15Y = (YF_KS11.loc[CAL_15Y] / YF_KS11.loc[CAL_15Y].iloc[0]) * ((1.0 + FINAL_CONFIG["KOSPI_EST_DIV_YIELD"]) ** yrs_15)
    NAV_B_15Y, NAV_C_15Y = build_bc_navs(CAL_15Y)

    # ① 보수적 15년 바구니 A: 2012-04~2017-04는 KOSPI TR 그대로, 2017-04부터 정밀 PIT(최종안 ②) 연결
    splice_dt = A_FINAL_9Y.index[0]
    pre_part = K_TR_15Y.loc[:splice_dt]
    post_part = A_FINAL_9Y.reindex(CAL_15Y[CAL_15Y >= splice_dt], method="ffill").bfill()
    post_scaled = post_part / post_part.iloc[0] * pre_part.iloc[-1]
    A_CONSERVATIVE_15Y = pd.concat([pre_part.iloc[:-1], post_scaled])

    # ② 참고용 Yahoo 20년 배당·저변동 프록시 (코스피 대표 장기 배당주 20종 중 매년 저변동+고배당 상위 8종 -> 2017-04 정밀 PIT 연결)
    PROXY_TKS = [
        "005930.KS", "005380.KS", "000270.KS", "105560.KS", "055550.KS",
        "086790.KS", "000810.KS", "032830.KS", "016360.KS", "029780.KS",
        "033780.KS", "017670.KS", "030200.KS", "032640.KS", "012750.KS",
        "030000.KS", "003550.KS", "005490.KS", "010130.KS", "086280.KS",
    ]
    def _fetch_yf_px_div(sym: str):
        try:
            r = requests.get(
                f"https://query1.finance.yahoo.com/v8/finance/chart/{sym}",
                params={"range": "20y", "interval": "1d", "events": "div"},
                headers={"User-Agent": "Mozilla/5.0 (dividend-final-bt)"},
                timeout=15,
            )
            res = r.json()["chart"]["result"][0]
            ts = res["timestamp"]
            adj = res["indicators"].get("adjclose", [{}])[0].get("adjclose") or res["indicators"]["quote"][0]["close"]
            raw = res["indicators"]["quote"][0]["close"]
            idx = [pd.Timestamp(datetime.fromtimestamp(t, tz=timezone.utc).strftime("%Y-%m-%d")) for t in ts]
            df = pd.DataFrame({"adj": adj, "raw": raw}, index=idx).dropna()
            df = df[~df.index.duplicated(keep="last")].sort_index()
            divs = res.get("events", {}).get("dividends", {})
            d_by_y = {}
            for k, v in divs.items():
                y = datetime.fromtimestamp(int(k), tz=timezone.utc).year
                d_by_y[y] = d_by_y.get(y, 0.0) + float(v.get("amount", 0.0))
            return sym, df, d_by_y
        except Exception:
            return sym, None, {}

    print("[15년 프록시 추정] Yahoo 코스피 대표 배당주 20종 (2011~2017 배당·저변동) 수집 중...")
    px_proxy, div_proxy = {}, {}
    with ThreadPoolExecutor(max_workers=8) as ex:
        for sym, df, dmap in ex.map(_fetch_yf_px_div, PROXY_TKS):
            if df is not None and len(df) > 500:
                px_proxy[sym] = df
                div_proxy[sym] = dmap

    # 2012-04 ~ 2017-04 프록시 수익곡선 (매년 4월 저변동성 + 배당수익률 50:50 상위 8종목 동일가중)
    pre_cal = CAL_15Y[CAL_15Y <= splice_dt]
    pre_rebal = [pre_cal[pre_cal >= pd.Timestamp(f"{y}-04-01")][0] for y in range(2012, 2017)]
    p_nav = np.ones(len(pre_cal), dtype=float)
    p_units = {}
    p_cash = 1.0
    for i, dt in enumerate(pre_cal):
        if dt in pre_rebal:
            tot = p_cash + sum(u * float(px_proxy[s].loc[:dt, "adj"].iloc[-1]) for s, u in p_units.items())
            cands = []
            for s, df in px_proxy.items():
                h = df.loc[:dt]
                if len(h) < 200:
                    continue
                vol = float(h["adj"].tail(252).pct_change().dropna().std() * np.sqrt(252))
                raw_px = float(h["raw"].iloc[-1])
                d_prev = div_proxy[s].get(dt.year - 1, 0.0)
                yld = d_prev / raw_px if raw_px > 0 else 0.0
                if yld > 0:
                    cands.append((s, vol, yld))
            if cands:
                cdf = pd.DataFrame(cands, columns=["s", "vol", "yld"])
                cdf["sc"] = 0.5 * (-cdf["vol"]).rank(pct=True) + 0.5 * cdf["yld"].rank(pct=True)
                chosen = cdf.nlargest(min(8, len(cdf)), "sc")["s"].tolist()
                tot_net = tot * (1.0 - FINAL_CONFIG["FEE_RATE"])
                p_units = {s: (tot_net / len(chosen)) / float(px_proxy[s].loc[:dt, "adj"].iloc[-1]) for s in chosen}
                p_cash = 0.0
        p_nav[i] = p_cash + sum(u * float(px_proxy[s].loc[:dt, "adj"].iloc[-1]) for s, u in p_units.items())
    pre_proxy_s = pd.Series(p_nav, index=pre_cal)
    post_proxy_scaled = post_part / post_part.iloc[0] * pre_proxy_s.iloc[-1]
    A_PROXY_15Y = pd.concat([pre_proxy_s.iloc[:-1], post_proxy_scaled])

    SPECS_15Y = [
        ("15년 보수추정 50/20/30 (12~17 KOSPI -> 17~ 최종안②)", A_CONSERVATIVE_15Y, (0.50, 0.20, 0.30)),
        ("15년 보수추정 40/15/45 (12~17 KOSPI -> 17~ 최종안②)", A_CONSERVATIVE_15Y, (0.40, 0.15, 0.45)),
        ("15년 프록시추정 50/20/30 (12~17 저변동배당 -> 17~ 최종안②)", A_PROXY_15Y, (0.50, 0.20, 0.30)),
        ("15년 프록시추정 40/15/45 (12~17 저변동배당 -> 17~ 최종안②)", A_PROXY_15Y, (0.40, 0.15, 0.45)),
        ("15년 국내 100% (KOSPI TR 추정)",                    K_TR_15Y,           (1.00, 0.00, 0.00)),
        ("15년 미국 100% (S&P500 2 : 배당다우 1 KRW)",         K_TR_15Y,           (0.00, 0.00, 1.00)),
    ]
    rows_15 = []
    for nm, na, wt in SPECS_15Y:
        ls = sim_3basket_lumpsum(na, NAV_B_15Y, NAV_C_15Y, wt)
        PORT_NAVS_15Y[nm] = ls
        mf = calc_perf(ls)
        m_est = calc_perf(ls, start_dt="2012-04-01", end_dt="2017-04-01")
        m_pit = calc_perf(ls, start_dt="2017-04-01")
        dca = sim_3basket_dca(na, NAV_B_15Y, NAV_C_15Y, wt)
        rows_15.append({
            "초장기 14.7년 추정 (2012-04 ~ 최신)": nm,
            "전체 CAGR": f"{mf['cagr']:+.2%}",
            "MDD": f"{mf['mdd']:.1%}",
            "샤프": f"{mf['sharpe']:.2f}",
            "추정구간(12~17)": f"{m_est['cagr']:+.2%}",
            "정밀PIT(17~26)": f"{m_pit['cagr']:+.2%}",
            "월200만 XIRR": f"{dca['xirr']:+.2%}",
            "적립 최종(투입3.5억)": f"{dca['final_val']/1e8:.2f}억",
        })
    DF_PORT_15Y = pd.DataFrame(rows_15)
    print("\n■ [검증 ③] 초장기 14.7년 추정 백테스트 (2012-04 ~ 최신, SCHD 상장 이후 전 구간)")
    display(DF_PORT_15Y)

In [ ]:
# ==============================================================================
# 8. 4패널 시각화 대시보드 + A4 PDF/HTML 리포트 & CSV Google Drive 자동 저장
# ==============================================================================
def export_html_to_pdf(html_str: str, pdf_path: str, chart_b64: str) -> bool:
    try:
        from weasyprint import HTML as WP_HTML
        WP_HTML(string=html_str).write_pdf(pdf_path)
        return True
    except Exception:
        pass
    try:
        from matplotlib.backends.backend_pdf import PdfPages
        import matplotlib.image as mpimg
        with PdfPages(pdf_path) as pdf:
            img = mpimg.imread(io.BytesIO(base64.b64decode(chart_b64)), format="png")
            fig_p = plt.figure(figsize=(11.69, 8.27))
            plt.imshow(img); plt.axis("off")
            plt.tight_layout(); pdf.savefig(fig_p, dpi=150); plt.close(fig_p)
        return True
    except Exception:
        return False

as_of_str = datetime.now(KST).strftime("%Y-%m-%d")
run_stamp = datetime.now(KST).strftime("%Y-%m-%d_%H%M%S")

fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15.6, 9.0), facecolor="white")
fig.suptitle("한국 배당성장주 최종 통합 검증 대시보드 (9.7년 클린 PIT + 14.7년 장기 추정)", fontsize=15.5, fontweight="bold", color="#0F2922", y=0.985)
fig.text(0.5, 0.946, f"기준일 {as_of_str}  |  {DATASET_LABEL} (버그 4종 보정)  |  최종안: 저변동·절대밸류·배당률·진성주주환원 + 상대70% 게이트 + 금융≤50%", ha="center", fontsize=9.2, color="#64748B")

# (1) 9.7Y Basket A 단독
a_plot_items = [
    ("최종안 ②: ① + 금융 ≤ 50% 상한 (권장)", "#059669", 2.5, "-"),
    ("현행 전략 A (클린 기준선)", "#1D4ED8", 1.8, "-"),
    ("원칙부합 전체 동일가중 (EW)", "#D97706", 1.5, "-"),
    ("KOSPI_TR", "#64748B", 1.7, "--"),
]
for k, col, lw, ls in a_plot_items:
    s = (A_NAVS_9Y[k] - 1.0) * 100
    lbl = "KOSPI TR 추정" if k == "KOSPI_TR" else k.split(":")[0]
    ax1.plot(s.index, s, label=f"{lbl} ({s.iloc[-1]:+.0f}%)", color=col, lw=lw, ls=ls)
ax1.set_title("① 바구니 A 단독 누적 수익률 (확장 9.7년 PIT, 2017-04~)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.legend(loc="upper left", fontsize=8.0, frameon=True, facecolor="white")
ax1.grid(True, color="#E2E8F0", ls="--", lw=0.7); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)

# (2) 9.7Y Integrated A+B+C
p_plot_items = [
    ("현행 50/20/30 (A=최종안 ② 권장)", "#0F766E", 2.4, "-"),
    ("권고 40/15/45 (A=최종안 ② 권장)", "#7C3AED", 2.4, "-"),
    ("국내 100% (KOSPI TR)", "#64748B", 1.5, "--"),
    ("미국 100% (S&P500 2 : 배당다우 1)", "#DC2626", 1.5, ":"),
]
for k, col, lw, ls in p_plot_items:
    s = (PORT_NAVS_9Y[k] - 1.0) * 100
    ax2.plot(s.index, s, label=f"{k.split('(')[0].strip()} ({s.iloc[-1]:+.0f}%)", color=col, lw=lw, ls=ls)
ax2.set_title("② A+B+C 통합 포트폴리오 (50/20/30 vs 40/15/45, 9.7년)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.legend(loc="upper left", fontsize=8.0, frameon=True, facecolor="white")
ax2.grid(True, color="#E2E8F0", ls="--", lw=0.7); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)

# (3) 14.7Y Extended Estimation
if PORT_NAVS_15Y:
    ax3.axvspan(pd.Timestamp("2012-04-01"), pd.Timestamp("2017-04-01"), color="#FEF3C7", alpha=0.45, label="추정 참고 구간 (12~17)")
    for k, col, lw, ls in [
        ("15년 보수추정 40/15/45 (12~17 KOSPI -> 17~ 최종안②)", "#7C3AED", 2.4, "-"),
        ("15년 보수추정 50/20/30 (12~17 KOSPI -> 17~ 최종안②)", "#0F766E", 2.2, "-"),
        ("15년 국내 100% (KOSPI TR 추정)", "#64748B", 1.5, "--"),
        ("15년 미국 100% (S&P500 2 : 배당다우 1 KRW)", "#DC2626", 1.5, ":"),
    ]:
        s = (PORT_NAVS_15Y[k] - 1.0) * 100
        ax3.plot(s.index, s, label=f"{k.split('(')[0].strip()} ({s.iloc[-1]:+.0f}%)", color=col, lw=lw, ls=ls)
    ax3.set_title("③ 초장기 14.7년 통합 추정 (2012-04~최신, 노란음영=추정구간)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
    ax3.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax3.legend(loc="upper left", fontsize=7.6, frameon=True, facecolor="white")
    ax3.grid(True, color="#E2E8F0", ls="--", lw=0.7); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)

# (4) Risk-Return Efficiency Map (9.7Y)
eff_pts = [
    ("A 단독 (전략A)", A_ORIG_9Y, "#1D4ED8", "o", 90),
    ("A 단독 (최종안②)", A_FINAL_9Y, "#059669", "s", 110),
    ("통합 50/20/30", PORT_NAVS_9Y["현행 50/20/30 (A=최종안 ② 권장)"], "#0F766E", "*", 230),
    ("통합 40/15/45", PORT_NAVS_9Y["권고 40/15/45 (A=최종안 ② 권장)"], "#7C3AED", "*", 250),
    ("국내 100%", K_TR_9Y, "#64748B", "D", 95),
    ("미국 100%", NAV_C_9Y, "#DC2626", "D", 95),
]
for lbl, s, col, mk, sz in eff_pts:
    m = calc_perf(s)
    ax4.scatter(m["vol"] * 100, m["cagr"] * 100, s=sz, c=col, marker=mk, edgecolors="#0F172A", lw=0.8, zorder=5)
    ax4.annotate(f"{lbl}\n(샤프 {m['sharpe']:.2f})", (m["vol"] * 100, m["cagr"] * 100), xytext=(6, 4), textcoords="offset points", fontsize=8.1, fontweight="bold" if mk == "*" else "normal")
ax4.set_title("④ 위험(변동성) 대비 수익률 효율 지도 (9.7년, 왼쪽 위 = 최고)", fontsize=11, fontweight="bold", color="#0F2922", loc="left")
ax4.set_xlabel("연환산 변동성 (%)"); ax4.set_ylabel("연평균 수익률 CAGR (%)")
ax4.grid(True, color="#E2E8F0", ls="--", lw=0.7); ax4.spines["top"].set_visible(False); ax4.spines["right"].set_visible(False)

fig.subplots_adjust(top=0.90, bottom=0.07, left=0.07, right=0.98, hspace=0.33, wspace=0.22)
buf = io.BytesIO()
fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor="white")
DASH_B64 = base64.b64encode(buf.getvalue()).decode()
if IN_COLAB:
    plt.show()
plt.close(fig)

def render_tbl(df: pd.DataFrame) -> str:
    if df.empty:
        return ""
    ths = "".join(f"<th>{escape(str(c))}</th>" for c in df.columns)
    trs = []
    for _, row in df.iterrows():
        tds = [f'<td class="{"col-left" if idx == 0 else ""}">{escape(str(row[c]))}</td>' for idx, c in enumerate(df.columns)]
        trs.append("<tr>" + "".join(tds) + "</tr>")
    return f'<table class="tbl"><thead><tr>{ths}</tr></thead><tbody>{"".join(trs)}</tbody></table>'

html_report = f"""<!doctype html>
<html lang="ko"><head><meta charset="utf-8"><title>최종 통합 포트폴리오 백테스트 ({as_of_str})</title>
<style>
  @page {{ size: A4 landscape; margin: 10mm; }}
  body {{ margin:0; padding:12px; background:#F8FAFC; color:#0F172A; font-family:'NanumGothic',sans-serif; font-size:10.5px; line-height:1.5; }}
  .wrap {{ max-width:1120px; margin:0 auto; }}
  .head {{ background:linear-gradient(135deg,#0F2922,#134E4A); color:#fff; padding:16px 20px; border-radius:10px; margin-bottom:12px; border-bottom:4px solid #D4A34A; }}
  .card {{ background:#fff; border:1px solid #E2E8F0; border-radius:10px; padding:12px 14px; margin-bottom:10px; break-inside:avoid; }}
  h1 {{ margin:0 0 4px; font-size:18px; }} h2 {{ margin:0 0 8px; font-size:12.5px; color:#0F2922; }}
  table.tbl {{ width:100%; border-collapse:collapse; font-size:9.3px; }}
  table.tbl th {{ background:#F1F5F9; padding:5px; border-top:1.5px solid #0F2922; border-bottom:1px solid #CBD5E1; text-align:right; }}
  table.tbl td {{ padding:5px; border-bottom:1px solid #EEF2F6; text-align:right; }}
  table.tbl td.col-left, table.tbl th:first-child {{ text-align:left; font-weight:700; }}
  img.chart {{ width:100%; height:auto; display:block; }}
  .page-break {{ page-break-before:always; break-before:page; }}
</style></head><body><div class="wrap">
  <div class="head">
    <h1>한국 배당성장주 최종 통합 포트폴리오 검증 리포트 ({as_of_str})</h1>
    <div>{escape(DATASET_LABEL)} · 버그 4종 보정 · 9.7년 확장 PIT + 14.7년 장기 추정 · 매월 적립식 &amp; 3바구니 분산 검증</div>
  </div>
  <div class="card"><h2>01. 4패널 최종 검증 대시보드</h2><img class="chart" src="data:image/png;base64,{DASH_B64}"></div>
  <div class="card page-break"><h2>02. 바구니 A 단독 비교 (확장 9.7년 PIT, 2017-04 ~ 최신)</h2>{render_tbl(DF_A_9Y)}</div>
  <div class="card"><h2>03. 바구니 A 단독 비교 (공식 7.7년 PIT, 2019-04 ~ 최신)</h2>{render_tbl(DF_A_7Y)}</div>
  <div class="card page-break"><h2>04. A+B+C 통합 포트폴리오 (확장 9.7년, 50/20/30 vs 40/15/45)</h2>{render_tbl(DF_PORT_9Y)}</div>
  <div class="card"><h2>05. 초장기 14.7년 추정 백테스트 (2012-04 ~ 최신, 12~17년 추정 한계 명시)</h2>{render_tbl(DF_PORT_15Y)}</div>
</div></body></html>"""

r_html = f"final_portfolio_report_{run_stamp}.html"
r_pdf = f"final_portfolio_report_{run_stamp}.pdf"
Path(r_html).write_text(html_report, encoding="utf-8")
pdf_ok = export_html_to_pdf(html_report, r_pdf, DASH_B64)
print(f"[리포트 완료] HTML={Path(r_html).stat().st_size/1024:.1f} KB | PDF={'성공' if pdf_ok else '건너뜀'}")

if DRIVE_REPORT_DIR is not None:
    try:
        (DRIVE_REPORT_DIR / r_html).write_text(html_report, encoding="utf-8")
        if pdf_ok and Path(r_pdf).exists():
            (DRIVE_REPORT_DIR / r_pdf).write_bytes(Path(r_pdf).read_bytes())
        DF_A_9Y.to_csv(DRIVE_REPORT_DIR / f"final_basketA_9y_{run_stamp}.csv", index=False, encoding="utf-8-sig")
        DF_PORT_9Y.to_csv(DRIVE_REPORT_DIR / f"final_portfolio_9y_{run_stamp}.csv", index=False, encoding="utf-8-sig")
        if not DF_PORT_15Y.empty:
            DF_PORT_15Y.to_csv(DRIVE_REPORT_DIR / f"final_portfolio_15y_{run_stamp}.csv", index=False, encoding="utf-8-sig")
        print(f"[Drive 저장 완료] {DRIVE_REPORT_DIR / r_pdf}")
    except Exception as e:
        print(f"[주의] Drive 저장 실패({type(e).__name__}): {e}")